In [1]:
import os
import pandas as pd
from pathlib import Path
import sys
import numpy as np
import re

df_train = pd.read_json("dataset/primevul_train.jsonl", lines=True, engine='pyarrow')
df_val = pd.read_json("dataset/primevul_valid.jsonl", lines=True, engine='pyarrow') 
df_test = pd.read_json("dataset/primevul_test.jsonl", lines=True, engine='pyarrow') 
df = pd.concat([df_train, df_val, df_test], ignore_index=True)

# 向上两级到项目根目录
PROJECT_ROOT = Path(os.path.abspath("")).resolve()
while not (PROJECT_ROOT / "configs").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

from src.utils.cwe import CWEUtils
from src.utils.cve import CVEUtils

MIN_CODE_LEN = 35  # 代码最短字符数（过滤过短的无意义代码）
MAX_CODE_LEN = 65535   # 代码最长字符数（过滤过长的异常代码）

print(len(df))
df.head()

Project root: <PROJECT_ROOT>
224533


,idx,project,commit_id,project_url,commit_url,commit_message,target,func,func_hash,file_name,file_hash,cwe,cve,cve_desc,nvd_url
0,0,openssl,ca989269a2876bae79393bd54c3e72d49975fc75,https://github.com/openssl/openssl,https://git.openssl.org/gitweb/?p=openssl.git;...,Use version in SSL_METHOD not SSL structure.\n...,1,long ssl_get_algorithm2(SSL *s)\n {\n ...,2.550877e+38,None,NaN,[CWE-310],CVE-2013-6449,The ssl_get_algorithm2 function in ssl/s3_lib....,https://nvd.nist.gov/vuln/detail/CVE-2013-6449
1,1,savannah,190cef6eed37d0e73a73c1e205eb31d45ab60a3c,https://git.savannah.gnu.org/gitweb/?p=gnutls,https://git.savannah.gnu.org/gitweb/?p=gnutls....,None,1,gnutls_session_get_data (gnutls_session_t sess...,2.660054e+38,None,NaN,[CWE-119],CVE-2011-4128,Buffer overflow in the gnutls_session_get_data...,https://nvd.nist.gov/vuln/detail/CVE-2011-4128
2,2,savannah,e82ef4545e9e98cbcb032f55d7c750b81e3a0450,https://git.savannah.gnu.org/gitweb/?p=gnutls,https://git.savannah.gnu.org/gitweb/?p=gnutls....,None,1,gnutls_session_get_data (gnutls_session_t sess...,1.626195e+38,None,NaN,[CWE-119],CVE-2011-4128,Buffer overflow in the gnutls_session_get_data...,https://nvd.nist.gov/vuln/detail/CVE-2011-4128
3,3,savannah,075d7556964f5a871a73c22ac4b69f5361295099,https://git.savannah.gnu.org/gitweb/?p=gnutls,https://git.savannah.gnu.org/cgit/wget.git/com...,None,1,"getftp (struct url *u, wgint passed_expected_b...",1.147531e+38,None,NaN,[CWE-200],CVE-2015-7665,Tails before 1.7 includes the wget program but...,https://nvd.nist.gov/vuln/detail/CVE-2015-7665
4,5,ghostscript,83d4dae44c71816c084a635550acc1a51529b881,http://git.ghostscript.com/?p=mupdf,http://git.ghostscript.com/?p=mupdf.git;a=comm...,None,1,void fz_init_cached_color_converter(fz_context...,1.831395e+38,colorspace.c,6.862522e+36,[CWE-20],CVE-2018-1000040,"In MuPDF 1.12.0 and earlier, multiple use of u...",https://nvd.nist.gov/vuln/detail/CVE-2018-1000040


In [2]:
# 只保留有漏洞的样本
df = df[df['target'] == 1] 

# 过滤掉"cwe"为null/空字符串的样本（PrimeVul 原始列名为 cwe）
df = df[
    df['cwe'].notna()
    & (df['cwe'].astype(str).str.strip() != '')
    & (df['cwe'].astype(str).str.strip() != 'None')
    & (df['cve'].notna())
    & (df['cve'].astype(str).str.strip() != '')
    & (df['cve'].astype(str).str.strip() != 'None')
    & (df['func'].notna())
    & (df['func'].astype(str).str.strip() != '')
    & (df['func'].astype(str).str.strip() != 'None')
]

# 将 cwe/cve 重命名为 cwe_id/cve_id（后续统一使用新列名）, 将'idx'列重命名为'id', 将'func'列重命名为'vul_code'
df = df.rename(columns={
    'cwe': 'cwe_id', 
    'cve': 'cve_id', 
    'idx': 'id', 
    'func': 'vul_code'
})

# 移除无关列
df = df.drop(
    columns=[
        'project', 'commit_id', 'project_url', 'commit_url', 'commit_message',
        'target', 'func_hash', 'file_name', 'file_hash', 'nvd_url'
    ]
)

# 过滤掉 vul_code 过短(<MIN_CODE_LEN) 或过长(>MAX_CODE_LEN) 的行
vul_len = df['vul_code'].str.len()
df = df[vul_len.between(MIN_CODE_LEN, MAX_CODE_LEN, inclusive='both')]

# 新增一个'source'列
df['source'] = 'PrimeVul'

print(len(df))
df.head()

5969


,id,vul_code,cwe_id,cve_id,cve_desc,source
0,0,long ssl_get_algorithm2(SSL *s)\n {\n ...,[CWE-310],CVE-2013-6449,The ssl_get_algorithm2 function in ssl/s3_lib....,PrimeVul
1,1,gnutls_session_get_data (gnutls_session_t sess...,[CWE-119],CVE-2011-4128,Buffer overflow in the gnutls_session_get_data...,PrimeVul
2,2,gnutls_session_get_data (gnutls_session_t sess...,[CWE-119],CVE-2011-4128,Buffer overflow in the gnutls_session_get_data...,PrimeVul
3,3,"getftp (struct url *u, wgint passed_expected_b...",[CWE-200],CVE-2015-7665,Tails before 1.7 includes the wget program but...,PrimeVul
4,5,void fz_init_cached_color_converter(fz_context...,[CWE-20],CVE-2018-1000040,"In MuPDF 1.12.0 and earlier, multiple use of u...",PrimeVul


In [3]:
# 将 cwe 列从 numpy 数组转换为字符串格式（提取第一个 CWE ID）
def extract_cwe_id(cwe_field):
    """从 numpy 数组或列表中提取第一个 CWE ID 作为字符串"""
    if cwe_field is None:
        return None
    if isinstance(cwe_field, np.ndarray) and len(cwe_field) > 0:
        return str(cwe_field[0])
    elif isinstance(cwe_field, list) and len(cwe_field) > 0:
        return str(cwe_field[0])
    elif isinstance(cwe_field, str):
        return cwe_field.strip()
    return None

df['cwe_id'] = df['cwe_id'].apply(extract_cwe_id)

In [4]:
# 新增并填充 'cwe_desc' 列
cwe_utils = CWEUtils(PROJECT_ROOT / "data/raw/cwe/cwec_v4.19.1.xml")

def get_cwe_desc(cwe_field):
    """根据 cwe 字段生成 CWE prompt 文本（使用 CWEUtils.get_prompt）。

    支持多种 cwe 格式：numpy数组、列表、字符串等。
    返回值为结构化的 prompt 文本；若无法解析/查询则返回 None。
    """
    if cwe_field is None:
        return None

    cwe_id = None

    # 处理 numpy 数组格式（如 array(['CWE-20'], dtype=object)）
    if isinstance(cwe_field, np.ndarray) and len(cwe_field) > 0:
        cwe_id = cwe_field[0]
    # 处理列表格式，如 ["CWE-20"] 或 ["CWE-20", "CWE-79"]
    elif isinstance(cwe_field, list) and len(cwe_field) > 0:
        cwe_id = cwe_field[0]
    # 处理字符串格式
    elif isinstance(cwe_field, str):
        match = re.search(r"CWE-\d+", cwe_field)
        if match:
            cwe_id = match.group()
        else:
            cwe_id = cwe_field.strip().strip("[]'\"")
            if "," in cwe_id:
                cwe_id = cwe_id.split(",")[0].strip().strip("'\"")

    if cwe_id is None or str(cwe_id).strip() == "":
        return None

    prompt = cwe_utils.get_prompt(str(cwe_id).strip())
    if "Result: The requested entry was not found in the CWE dataset." in prompt:
        return None

    return prompt

# 填充 cwe_desc 列
df['cwe_desc'] = df['cwe_id'].apply(get_cwe_desc)

# 验证结果
print(f"总样本数: {len(df)}")
print(f"cwe_desc 为空的样本数: {df['cwe_desc'].isna().sum()}")
print(f"cwe_desc 填充率: {(1 - df['cwe_desc'].isna().sum() / len(df)) * 100:.2f}%")
df.head()


总样本数: 5969
cwe_desc 为空的样本数: 0
cwe_desc 填充率: 100.00%


,id,vul_code,cwe_id,cve_id,cve_desc,source,cwe_desc
0,0,long ssl_get_algorithm2(SSL *s)\n {\n ...,CWE-310,CVE-2013-6449,The ssl_get_algorithm2 function in ssl/s3_lib....,PrimeVul,- CWE ID: CWE-310\n- Name: Cryptographic Issue...
1,1,gnutls_session_get_data (gnutls_session_t sess...,CWE-119,CVE-2011-4128,Buffer overflow in the gnutls_session_get_data...,PrimeVul,- CWE ID: CWE-119\n- Name: Improper Restrictio...
2,2,gnutls_session_get_data (gnutls_session_t sess...,CWE-119,CVE-2011-4128,Buffer overflow in the gnutls_session_get_data...,PrimeVul,- CWE ID: CWE-119\n- Name: Improper Restrictio...
3,3,"getftp (struct url *u, wgint passed_expected_b...",CWE-200,CVE-2015-7665,Tails before 1.7 includes the wget program but...,PrimeVul,- CWE ID: CWE-200\n- Name: Exposure of Sensiti...
4,5,void fz_init_cached_color_converter(fz_context...,CWE-20,CVE-2018-1000040,"In MuPDF 1.12.0 and earlier, multiple use of u...",PrimeVul,- CWE ID: CWE-20\n- Name: Improper Input Valid...


In [5]:
# 更新 'cve_desc' 列
cve_utils = CVEUtils(PROJECT_ROOT / "data/raw/cwe/CVE_v2026.03.18.json")

def update_cve_desc_with_prompt(cve_field):
    """
    使用 CVEUtils.get_prompt 方法获取 CVE 的完整上下文描述。
    如果 CVE ID 为空或找不到，则返回原有的 cve_desc 值。
    """
    if pd.isna(cve_field) or str(cve_field).strip() == "":
        return None
    
    # 使用 get_prompt 获取结构化的 CVE 上下文
    prompt_text = cve_utils.get_prompt(str(cve_field).strip())
    
    # 检查是否返回了 "not found" 的错误信息
    if "Result: This entry was not found in the CVE dataset." in prompt_text:
        return None
    
    return prompt_text

# 使用 get_prompt 方法更新 cve_desc 列
df['cve_desc'] = df['cve_id'].apply(update_cve_desc_with_prompt)

# 统计更新结果
found_count = df['cve_desc'].notna().sum()
not_found_count = df['cve_desc'].isna().sum()
print(f"更新完成：")
print(f"  - 成功获取 CVE 描述的样本数: {found_count}")
print(f"  - 未找到 CVE 描述的样本数: {not_found_count}")
print(f"  - 覆盖率: {found_count / len(df) * 100:.2f}%")
df.head()

更新完成：
  - 成功获取 CVE 描述的样本数: 5969
  - 未找到 CVE 描述的样本数: 0
  - 覆盖率: 100.00%


,id,vul_code,cwe_id,cve_id,cve_desc,source,cwe_desc
0,0,long ssl_get_algorithm2(SSL *s)\n {\n ...,CWE-310,CVE-2013-6449,- CVE ID: CVE-2013-6449\n- Status: Deferred\n-...,PrimeVul,- CWE ID: CWE-310\n- Name: Cryptographic Issue...
1,1,gnutls_session_get_data (gnutls_session_t sess...,CWE-119,CVE-2011-4128,- CVE ID: CVE-2011-4128\n- Status: Deferred\n-...,PrimeVul,- CWE ID: CWE-119\n- Name: Improper Restrictio...
2,2,gnutls_session_get_data (gnutls_session_t sess...,CWE-119,CVE-2011-4128,- CVE ID: CVE-2011-4128\n- Status: Deferred\n-...,PrimeVul,- CWE ID: CWE-119\n- Name: Improper Restrictio...
3,3,"getftp (struct url *u, wgint passed_expected_b...",CWE-200,CVE-2015-7665,- CVE ID: CVE-2015-7665\n- Status: Deferred\n-...,PrimeVul,- CWE ID: CWE-200\n- Name: Exposure of Sensiti...
4,5,void fz_init_cached_color_converter(fz_context...,CWE-20,CVE-2018-1000040,- CVE ID: CVE-2018-1000040\n- Status: Modified...,PrimeVul,- CWE ID: CWE-20\n- Name: Improper Input Valid...


In [6]:
# 将数据以jsonl的格式保存到data/raw/vul/Primevul.jsonl
output_path = PROJECT_ROOT / "data/raw/vul/Primevul.jsonl"
output_path.parent.mkdir(parents=True, exist_ok=True)

df.to_json(
    output_path,
    orient="records",
    lines=True,
    force_ascii=False
)

print(f"保存完成: {output_path} （共 {len(df)} 条样本）")

保存完成: <PROJECT_ROOT>/data/raw/vinj/Primevul.jsonl （共 5969 条样本）


In [7]:
# 统计每条数据的 cwe 列中包含多个 cwe 条目的数据个数
def cwe_len(x):
    # 情况1：已经是列表，如 ["CWE-20", "CWE-79"]
    if isinstance(x, list):
        return len(x)
    # 情况2：是字符串，用逗号/空格分隔（以防万一）
    if isinstance(x, str):
        parts = [p.strip() for p in x.split(",") if p.strip()]
        return len(parts)
    return 0

# 统计包含多个 CWE 的样本个数
cwe_counts = df["cwe_id"].apply(cwe_len)
multi_cwe_mask = cwe_counts > 1
print("包含多个 CWE 的样本个数:", multi_cwe_mask.sum())

# 统计dataframe中cve_desc为空/NaN的样本个数
print(f"cve_desc为空/NaN的样本个数: {df[df['cve_desc'].isna() | (df['cve_desc'].astype(str).str.strip() == '')].shape[0]}")

# 统计dataframe中cwe_desc为空/NaN的样本个数
print(f"cwe_desc为空/NaN的样本个数: {df[df['cwe_desc'].isna() | (df['cwe_desc'].astype(str).str.strip() == '')].shape[0]}")

# 统计dataframe中vul_code为空/NaN的样本个数
print(f"vul_code为空/NaN的样本个数: {df[df['vul_code'].isna() | (df['vul_code'].astype(str).str.strip() == '')].shape[0]}")

包含多个 CWE 的样本个数: 0
cve_desc为空/NaN的样本个数: 0
cwe_desc为空/NaN的样本个数: 0
vul_code为空/NaN的样本个数: 0
